# Chapter 12. Dynamic models and rhythm sensitivity

**Learning objective.** Compare a naive aggregate smooth with a model that represents speaker, item, token, and serial dependence, then stress-test duration-based rhythm summaries across speakers, material, and segmentation.

**Estimated time:** 90–120 minutes. **Exercises:** 12.3 and 12.4. Model estimates and rhythm metrics are generated teaching results, not empirical language classifications.

## Input and output contract

The notebook requires the preceding prepared trajectory, segmented interval data, and a prespecified comparison file. It writes population-level prediction grids, token-level residual-dependence summaries, rhythm metrics by speaker and sentence set, sensitivity results by material and segmentation, a trajectory gate, diagnostics, and provenance. Frame count is never reported as independent replication.

In [ ]:
find_repository <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "companion", "data", "MANIFEST.tsv"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Run from the repository root or a subdirectory.")
    current <- parent
  }
}
REPOSITORY_ROOT <- find_repository()
DATA_ROOT <- file.path(REPOSITORY_ROOT, "companion", "data", "ch12", "synthetic_prosody")
if (!dir.exists(DATA_ROOT)) stop("Chapter 12 fixture absent. Run companion/scripts/generate_ch12_fixture.py explicitly; no data are substituted silently.")
source(file.path(REPOSITORY_ROOT, "companion", "r", "ch12.R"))
cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated contours and intervals, not human prosody.
")

In [ ]:
PREPARED <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch12", "01_pitch_trajectory_preparation", "trajectory_preprocessing.tsv")
if (!file.exists(PREPARED)) stop("Run 01_pitch_trajectory_preparation.ipynb first; the model notebook will not substitute a trajectory table.")
OUTPUT <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch12", "02_dynamic_models_and_rhythm")
result <- run_dynamic_models_and_rhythm(PREPARED, file.path(DATA_ROOT, "segmented_intervals.tsv"), file.path(DATA_ROOT, "model_comparison.yaml"), OUTPUT)
stopifnot(nrow(result$predictions) == 204)
stopifnot(nrow(result$residual_dependence) == 2)
stopifnot(!result$gate$frame_count_is_replication_count)
print(result$residual_dependence)
print(head(result$rhythm))

## Learner gate

Compare residual lag-one dependence and uncertainty between the naive and structured models. Explain why thousands of frames still derive from only twelve speakers, eight items, and ninety-six tokens. Compare rhythm metrics by speaker, sentence subset, and segmentation profile, then decide whether the evidence permits only a sample description, a synthetic group contrast, or any broader classification.

In [ ]:
stopifnot(all(file.exists(file.path(OUTPUT, c("dynamic_model_diagnostics.html", "trajectory_predictions.tsv", "residual_dependence.tsv", "rhythm_metrics_by_unit.tsv", "rhythm_sensitivity.tsv", "model_warnings.tsv", "trajectory_gate.yaml", "run_provenance.json")))))
print(result$gate)